In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import numpy.linalg as lin
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.font_manager import FontProperties

In [2]:
plt.rcParams["font.family"] = "Arial"

In [3]:
def calcbo(LRdist,HRdist,bincrates):
    scH=np.cumsum(HRdist)/np.sum(HRdist)
    scL=np.cumsum(LRdist)/np.sum(LRdist)
    rH = np.interp(0.5, scH, bincrates)
    rL = np.interp(0.5, scL, bincrates)
    bo = 100*((rH-rL)/rL)

    db=(bincrates[2]-bincrates[1])/bincrates[1]  
    dpamt=HRdist-LRdist
    # calculate dp/dln(r)
    nb=len(pamtL);
    dpdlnrc= np.concatenate((np.array([0]),LRdist[2:nb]-LRdist[0:(nb-2)],np.array([0])))/(2*db) # centered difference (2nd order accurate; now used near endpoints)
    prm2=LRdist[:(nb-4)]
    prm1=LRdist[1:(nb-3)]
    prp1=LRdist[3:(nb-1)]
    prp2=LRdist[4:]
    dpdlnr= np.concatenate((np.array([0]),np.array([0]),8*(prp1-prm1)-(prp2-prm2),np.array([0]),np.array([0])))/(12*db) # 4th order accurate 
    dpdlnr[1]=dpdlnrc[1] # fill in with second order accurate near endpoints
    dpdlnr[nb-2]=dpdlnrc[nb-2] 

    pamtbo=-(bo/100)*dpdlnr+LRdist
    err=100*np.nansum(np.abs(pamtbo[2:]-HRdist[2:]))/np.nansum(np.abs(dpamt[2:]))
    return (bo,pamtbo, err)

In [4]:
def shiftcalc(LRdist,HRdist,bincrates):


    db=(bincrates[2]-bincrates[1])/bincrates[1]  
    # calculate dp/dln(r)
    nb=len(LRdist);
    dpdlnrc= np.concatenate((np.array([0]),LRdist[2:nb]-LRdist[0:(nb-2)],np.array([0])))/(2*db) # centered difference (2nd order accurate; now used near endpoints)
    prm2=LRdist[:(nb-4)]
    prm1=LRdist[1:(nb-3)]
    prp1=LRdist[3:(nb-1)]
    prp2=LRdist[4:]
    dpdlnr= np.concatenate((np.array([0]),np.array([0]),8*(prp1-prm1)-(prp2-prm2),np.array([0]),np.array([0])))/(12*db) # 4th order accurate 
    dpdlnr[1]=dpdlnrc[1] # fill in with second order accurate near endpoints
    dpdlnr[nb-2]=dpdlnrc[nb-2] 

    dt=1
    
    dpamt=HRdist-LRdist
    # calculate the terms in the matrices that will need to be solved
    # (equation 10 in Pendergrass and Hartmann 2014, Two Modes of Change of the Distribution of Rain) 
    # calculate the terms in the matrices that will need to be solved
    # (equation 10 in Pendergrass and Hartmann 2014, Two Modes of Change of the Distribution of Rain) 
    spr2=np.nansum(LRdist**2)
    sdpdlnr2=np.nansum(dpdlnr**2)
    spdpdlnr=np.nansum(LRdist*dpdlnr)
    spdpm=np.nansum(LRdist*dpamt)
    sdpdlnrdpm=np.nansum(dpdlnr*dpamt)
    # set up the matrices
    B=np.array([[0],[-sdpdlnrdpm]])
    A=np.array([[0,-spdpdlnr],[0,sdpdlnr2]])

    try:
        x = np.linalg.solve(A, B)
    except np.linalg.LinAlgError:
        #print("Matrix is singular, using pseudo-inverse.")
        x = np.linalg.pinv(A) @ B
    shift=100*x[1] # shift mode (%)
    pamtshift=-x[1]*dpdlnr+LRdist
    err=100*np.nansum(np.abs(pamtshift[2:]-HRdist[2:]))/np.nansum(np.abs(dpamt[2:]))

    return (shift,pamtshift,err)

In [5]:
res=[10,25,100,200,500,1000]
n=0
LRc=[]

sm=[];si=[]
for i in range(1,len(res)):
    n=n+1
    data1= xr.open_dataset('./../data/raindist_data_10_'+str(res[i])+'km.nc')## Reading in data. Current structure is that the high resolution distribution is available in both input files.
    pamtH = data1.pamtH[1:].values ## rain amount of HR dataset
    pamtL= data1.pamtL[1:].values ## rain amount of LR dataset
    ppdfH= data1.ppdfH[1:].values ## rain frequency of HR dataset
    ppdfL=data1.ppdfL[1:].values ## rain frequency of LR dataset
    bincrates=data1.precipitation_bin[1:].values ## rain rate bins
    omodes=calcbo(pamtL,pamtH,bincrates)
    modes = shiftcalc(pamtL,pamtH,bincrates)

    sm.append(modes[0])
    si.append(omodes[0])
    LRc.append(str(res[i]/100)+'º')

shiftm=np.concatenate(sm)
#shifti=np.concatenate(si)
#LR=np.concatenate(LRc)
df=pd.DataFrame({'LR' :LRc,'b' : np.round(shiftm,2),'bo' : np.round(si,2)})


In [6]:
df

,LR,b,bo
0,0.25º,2.92,3.39
1,1.0º,14.08,16.02
2,2.0º,25.12,31.76
3,5.0º,41.72,75.00
4,10.0º,45.58,144.79


In [7]:
# AMS standard figure widths in inches
ams_sizes = {
    "one_column": 3.2,
    "two_thirds": 4.5,
    "two_columns": 5.5,
    "more_than_two_columns": 6.5,
}

# Common aspect ratio: width : height = 4:3 → height = width * 0.75
# You can change this based on the visual needs of your plot
aspect_ratio = 1


width = ams_sizes["one_column"]
height = width * aspect_ratio
colrs1=['#a6bddb','#74a9cf','#3690c0','#0570b0','#034e7b']

plt.figure(figsize=(width,height), constrained_layout=True)
plt.clf()
ax=plt.subplot(111)
sns.scatterplot(x='bo',y='b',s=100,data=df,hue="LR",palette=colrs1[::-1])
plt.xlim([-1,151])
plt.ylim([-1,151])
plt.xlabel('Shift Indicator $b_o$ (%)',fontsize=12,fontweight="bold")
plt.ylabel('Shift Mode b (%)',fontsize=12,fontweight="bold")
plt.xticks(fontsize=10,weight='bold')
plt.yticks(fontsize=10,weight='bold')
lims = [
            np.min([ax.get_xlim(), ax.get_ylim()]),  # min of both axes
            np.max([ax.get_xlim(), ax.get_ylim()]),  # max of both axes
        ]

# now plot both limits against eachother
ax.plot(lims, lims, 'grey', alpha=0.75)
ax.set_aspect('equal')
ax.set_xlim(lims)
ax.set_ylim(lims)
bold_font = FontProperties(weight='bold', size=10)
plt.legend(prop=bold_font)
filename='./../figs/figA1_bo_b.pdf'
plt.savefig(filename, format="pdf", bbox_inches="tight",dpi=300)
plt.close()